- **2A Calibration-grade evaluation**: ECE, NLL/log-loss, Brier, reliability diagrams, and selective prediction (coverage vs confidence).
- **2B Distribution shift stress tests**: (i) regime split (e.g., train 2022, test 2023--2024) and (ii) rolling-window evaluation (train 12 months, test 3 months, slide forward).




In [ ]:
# Install dependencies (Colab)
!pip -q install numpy pandas yfinance pennylane pennylane-lightning scikit-learn scipy matplotlib tqdm


In [ ]:
# Mount Google Drive and set output directory
from google.colab import drive
from datetime import datetime
import os

drive.mount('/content/drive')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
BASE_DIR = '/content/drive/MyDrive/Uncertainty_Resource/step2/' + timestamp
TABLE_DIR = os.path.join(BASE_DIR, 'tables')
FIG_DIR = os.path.join(BASE_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Saving outputs to:', BASE_DIR)


In [ ]:
# Imports
import numpy as np
import pandas as pd
import yfinance as yf
import pennylane as qml

import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

from scipy.stats import t
from tqdm.auto import tqdm


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss

def compute_metrics(y_true, y_prob, n_bins=10):
    """
    Returns: dict with accuracy, roc_auc, nll, brier, ece
    """
    y_true = np.asarray(y_true).reshape(-1)
    y_prob = np.asarray(y_prob).reshape(-1)

    # Hard predictions
    y_pred = (y_prob >= 0.5).astype(int)

    # Core metrics
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    nll = log_loss(y_true, y_prob, labels=[0, 1])
    brier = brier_score_loss(y_true, y_prob)

    # ECE (Expected Calibration Error)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        mask = bin_ids == b
        if np.any(mask):
            conf = y_prob[mask].mean()
            acc_bin = y_true[mask].mean()
            ece += np.abs(acc_bin - conf) * (mask.sum() / len(y_true))

    return {
        "accuracy": float(acc),
        "roc_auc": float(auc),
        "nll": float(nll),
        "brier": float(brier),
        "ece": float(ece),
    }


In [ ]:
# Metrics + plots: ECE, Brier, CI, reliability diagram, selective prediction

def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.mean()) * abs(acc - conf)
    return float(ece)


def brier_score(y_true, y_prob):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob)
    return float(np.mean((y_prob - y_true) ** 2))


def mean_ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    m = float(np.mean(x))
    if n <= 1:
        return m, 0.0
    se = float(np.std(x, ddof=1) / np.sqrt(n))
    h = float(t.ppf(0.975, n-1) * se)
    return m, h


def reliability_curve(y_true, y_prob, n_bins=10):
    # Returns bin_centers, accuracy_per_bin, confidence_per_bin, counts
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    accs, confs, centers, counts = [], [], [], []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            accs.append(np.nan)
            confs.append(np.nan)
            counts.append(0)
        else:
            accs.append(y_true[mask].mean())
            confs.append(y_prob[mask].mean())
            counts.append(int(mask.sum()))
        centers.append((lo + hi) / 2)
    return np.array(centers), np.array(accs), np.array(confs), np.array(counts)


def plot_reliability(ax, y_true, y_prob, label, n_bins=10):
    centers, accs, confs, counts = reliability_curve(y_true, y_prob, n_bins=n_bins)
    ax.plot([0, 1], [0, 1], linestyle='--')
    ax.plot(confs, accs, marker='o', label=label)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Empirical accuracy')


def selective_prediction_curve(y_true, y_prob, taus=None):
    # Keep predictions with confidence >= tau, where confidence = max(p, 1-p)
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    conf = np.maximum(y_prob, 1.0 - y_prob)
    pred = (y_prob >= 0.5).astype(int)
    if taus is None:
        taus = np.linspace(0.5, 0.95, 10)
    out = []
    for tau in taus:
        m = conf >= tau
        coverage = float(np.mean(m))
        if m.sum() == 0:
            acc = np.nan
        else:
            acc = float(np.mean(pred[m] == y_true[m]))
        out.append((float(tau), coverage, acc))
    return pd.DataFrame(out, columns=['tau', 'coverage', 'accuracy'])


In [ ]:
# Data prep: features + labels (directional next-day move)

class StockPredictor:
    def __init__(self, ticker='AAPL'):
        self.ticker = ticker

    def download(self, start, end):
        df = yf.download(self.ticker, start=start, end=end, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.dropna().copy()
        df.index = pd.to_datetime(df.index)
        return df

    def build_features(self, df):
        # Basic indicator set (lightweight + stable)
        close = df['Close']
        ret = close.pct_change()
        logret = np.log(close).diff()

        ma5 = close.rolling(5).mean()
        ma10 = close.rolling(10).mean()
        ma20 = close.rolling(20).mean()
        vol10 = ret.rolling(10).std()
        mom10 = close.pct_change(10)

        # RSI(14)
        delta = close.diff()
        up = delta.clip(lower=0)
        down = -delta.clip(upper=0)
        roll_up = up.rolling(14).mean()
        roll_down = down.rolling(14).mean()
        rs = roll_up / (roll_down + 1e-12)
        rsi14 = 100 - (100 / (1 + rs))

        X = pd.DataFrame({
            'ret': ret,
            'logret': logret,
            'ma5_gap': (close - ma5) / (ma5 + 1e-12),
            'ma10_gap': (close - ma10) / (ma10 + 1e-12),
            'ma20_gap': (close - ma20) / (ma20 + 1e-12),
            'vol10': vol10,
            'mom10': mom10,
            'rsi14': rsi14 / 100.0,
        }, index=df.index)

        # Label: next-day up/down
        y = (ret.shift(-1) > 0).astype(int)

        data = X.join(y.rename('y')).dropna().copy()
        return data

    def get_xy_by_dates(self, start, end):
        df = self.download(start, end)
        data = self.build_features(df)
        X = data.drop(columns=['y'])
        y = data['y']
        return X, y


In [ ]:
# Quantum reservoir features (shots + structure modes)

class QuantumReservoir:
    def __init__(self, n_qubits=6, seed=0, shots=None, mode='full'):
        self.n_qubits = n_qubits
        self.seed = int(seed)
        self.shots = shots  # None => analytic (infinite shots)
        self.mode = mode
        self.rng = np.random.default_rng(self.seed)

        self.dev = qml.device('default.qubit', wires=self.n_qubits, shots=self.shots)

        # fixed random parameters (untrained reservoir)
        self.w = self.rng.normal(scale=0.8, size=(self.n_qubits,))
        self.phi = self.rng.uniform(0, 2*np.pi, size=(self.n_qubits,))

    def _entangle(self):
        if self.mode == 'no_entanglement':
            return
        # simple all-to-all ring entanglement (lightweight)
        for i in range(self.n_qubits - 1):
            qml.CNOT(wires=[i, i+1])
        qml.CNOT(wires=[self.n_qubits - 1, 0])

    def _phase_scramble(self):
        if self.mode != 'phase_scramble':
            return
        # random Z phases to destroy coherent interference patterns
        for i in range(self.n_qubits):
            qml.RZ(self.rng.uniform(0, 2*np.pi), wires=i)

    def circuit(self, x):
        # Encode features (truncate/pad)
        x = np.asarray(x, dtype=float)
        if len(x) < self.n_qubits:
            x = np.pad(x, (0, self.n_qubits - len(x)))
        x = x[:self.n_qubits]

        for i in range(self.n_qubits):
            qml.RY(self.w[i] * x[i] + self.phi[i], wires=i)
        self._phase_scramble()
        self._entangle()

        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

    def transform(self, X):
        qnode = qml.QNode(lambda x: self.circuit(x), self.dev)
        feats = [qnode(row) for row in np.asarray(X)]
        return np.asarray(feats, dtype=float)


In [ ]:
# Readout training + evaluation

def fit_and_eval(X_train_feat, y_train, X_test_feat, y_test, seed=0):
    clf = LogisticRegression(max_iter=2000, solver='lbfgs', random_state=int(seed))
    clf.fit(X_train_feat, y_train)
    y_prob = clf.predict_proba(X_test_feat)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    out = {}
    out['accuracy'] = float(accuracy_score(y_test, y_pred))
    out['roc_auc'] = float(roc_auc_score(y_test, y_prob))
    out['nll'] = float(log_loss(y_test, y_prob, labels=[0,1]))
    out['brier'] = float(brier_score(y_test, y_prob))
    out['ece'] = float(expected_calibration_error(y_test, y_prob, n_bins=10))
    return out, y_prob


In [ ]:
# Step 2A/2B experiment runner: regime split + rolling windows

# ---- Config ----
TICKER = 'AAPL'
N_QUBITS = 6
K_REPS = 10

# shot setting: use the Step 1 best shot by default
SHOTS_BEST = 200

CONDITIONS = {
    'quantum_full': dict(mode='full', shots=SHOTS_BEST),
    'noise_matched': dict(mode='full', shots=SHOTS_BEST),  # handled specially
    'no_entanglement': dict(mode='no_entanglement', shots=SHOTS_BEST),
    'phase_scramble': dict(mode='phase_scramble', shots=SHOTS_BEST),
}

# Regime split: train 2022, test 2023-2024 (edit as needed)
TRAIN_START, TRAIN_END = '2022-01-01', '2023-01-01'
TEST_START, TEST_END   = '2023-01-01', '2025-01-01'

# Rolling windows
TRAIN_MONTHS = 12
TEST_MONTHS = 3
STEP_MONTHS = 3

predictor = StockPredictor(TICKER)


# ---- Shape utilities (robust to pandas column-vectors) ----

def _to_1d(a):
    a = np.asarray(a)
    if a.ndim == 2 and a.shape[1] == 1:
        return a.reshape(-1)
    if a.ndim == 2 and a.shape[1] == 2:
        return a[:, 1].reshape(-1)
    return a.reshape(-1)


def noise_matched_features(X_det_train, X_det_test, predictor_variance):
    # Diagonal variance match
    var = np.maximum(predictor_variance, 1e-12)
    std = np.sqrt(var)
    eps_train = np.random.normal(scale=std, size=X_det_train.shape)
    eps_test  = np.random.normal(scale=std, size=X_det_test.shape)
    return X_det_train + eps_train, X_det_test + eps_test


def split_and_scale(X, y, train_end_date):
    idx = X.index
    train_mask = idx < pd.to_datetime(train_end_date)
    X_train, y_train = X.loc[train_mask], y.loc[train_mask]
    X_test, y_test = X.loc[~train_mask], y.loc[~train_mask]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train.values)
    X_test_s = scaler.transform(X_test.values)
    return X_train_s, y_train.values, X_test_s, y_test.values


def run_single_split(train_start, train_end, test_end, condition, rep_seed):
    # Download a continuous range covering train+test so features align
    X, y = predictor.get_xy_by_dates(train_start, test_end)
    X_train, y_train, X_test, y_test = split_and_scale(X, y, train_end)

    # Ensure labels are 1-D for sklearn/pandas compatibility
    y_train = _to_1d(y_train)
    y_test  = _to_1d(y_test)

    # Build reservoir features
    if condition == 'noise_matched':
        # deterministic features
        res_det = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed, shots=None, mode='full')
        F_train_det = res_det.transform(X_train)
        F_test_det  = res_det.transform(X_test)

        # estimate per-feature variance induced by shots
        # (repeat a few times to estimate variance robustly)
        V_runs = []
        for s in range(5):
            res_s = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed + 1000 + s, shots=SHOTS_BEST, mode='full')
            F_train_s = res_s.transform(X_train)
            V_runs.append(np.var(F_train_s, axis=0, ddof=1))
        var_diag = np.mean(np.stack(V_runs, axis=0), axis=0)

        F_train, F_test = noise_matched_features(F_train_det, F_test_det, var_diag)
    else:
        cfg = CONDITIONS[condition]
        res = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed, shots=cfg['shots'], mode=cfg['mode'])
        F_train = res.transform(X_train)
        F_test  = res.transform(X_test)

    metrics, y_prob = fit_and_eval(F_train, y_train, F_test, y_test, seed=rep_seed)
    y_prob = _to_1d(y_prob)
    return metrics, (y_test, y_prob)


def make_rolling_folds(start, end, train_months=12, test_months=3, step_months=3):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    folds = []
    t0 = start
    while True:
        train_start = t0
        train_end = train_start + pd.DateOffset(months=train_months)
        test_end = train_end + pd.DateOffset(months=test_months)
        if test_end > end:
            break
        folds.append((train_start.date().isoformat(), train_end.date().isoformat(), test_end.date().isoformat()))
        t0 = t0 + pd.DateOffset(months=step_months)
    return folds





In [ ]:
# ---- Run regime split ----
regime_rows = []
regime_probs = []  # store for reliability/selection curves

for cond in CONDITIONS.keys():
    for k in tqdm(range(K_REPS), desc=f'Regime split ({cond})'):
        seed = 1234 + 10*k
        m, (y_t, y_p) = run_single_split(TRAIN_START, TRAIN_END, TEST_END, cond, seed)
        regime_rows.append({
            'scenario': 'regime',
            'condition': cond,
            'rep': k,
            'train_start': TRAIN_START,
            'train_end': TRAIN_END,
            'test_end': TEST_END,
            **m
        })
        regime_probs.append({'condition': cond, 'rep': k, 'y_true': y_t, 'y_prob': y_p})

regime_df = pd.DataFrame(regime_rows)
regime_df.to_csv(os.path.join(TABLE_DIR, 'regime_results_raw.csv'), index=False)

# ---- Run rolling windows ----
ROLL_START, ROLL_END = TRAIN_START, TEST_END
folds = make_rolling_folds(ROLL_START, ROLL_END, TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS)
print('Rolling folds:', len(folds))

rolling_rows = []
rolling_probs = []

for (tr_s, tr_e, te_e) in tqdm(folds, desc='Rolling folds'):
    for cond in CONDITIONS.keys():
        for k in range(max(1, K_REPS//5)):
            seed = 9000 + hash((tr_s, cond, k)) % 100000
            m, (y_t, y_p) = run_single_split(tr_s, tr_e, te_e, cond, seed)
            rolling_rows.append({
                'scenario': 'rolling',
                'fold_train_start': tr_s,
                'fold_train_end': tr_e,
                'fold_test_end': te_e,
                'condition': cond,
                'rep': k,
                **m
            })
            rolling_probs.append({'fold': (tr_s,tr_e,te_e), 'condition': cond, 'rep': k, 'y_true': y_t, 'y_prob': y_p})

rolling_df = pd.DataFrame(rolling_rows)
rolling_df.to_csv(os.path.join(TABLE_DIR, 'rolling_results_raw.csv'), index=False)

print('Saved raw tables.')

In [ ]:
# Summaries (mean ± 95% CI) for regime + rolling

def summarize(df, group_cols):
    rows=[]
    for keys, g in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys=(keys,)
        row={col: val for col,val in zip(group_cols, keys)}
        for metric in ['accuracy','roc_auc','nll','brier','ece']:
            m,h=mean_ci95(g[metric].values)
            row[f'{metric}_mean']=m
            row[f'{metric}_ci95']=h
        rows.append(row)
    return pd.DataFrame(rows)

regime_sum = summarize(regime_df, ['scenario','condition'])
rolling_sum = summarize(rolling_df, ['scenario','condition'])

summary = pd.concat([regime_sum, rolling_sum], ignore_index=True)
summary.to_csv(os.path.join(TABLE_DIR, 'summary_ci95_step2.csv'), index=False)

print('Saved:', os.path.join(TABLE_DIR, 'summary_ci95_step2.csv'))
display(summary)


In [ ]:
# Visualisations: reliability diagrams + selective prediction curves

# --- Reliability diagrams (regime) ---
fig, ax = plt.subplots(1, 1, figsize=(6,6))
for cond in CONDITIONS.keys():
    # concatenate across reps
    ys = np.concatenate([p['y_true'] for p in regime_probs if p['condition']==cond])
    ps = np.concatenate([p['y_prob'] for p in regime_probs if p['condition']==cond])
    plot_reliability(ax, ys, ps, label=cond, n_bins=10)
ax.legend()
ax.set_title('Reliability diagram (regime split)')
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'reliability_regime.png')
fig.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

# --- Selective prediction curves (regime) ---
fig, ax = plt.subplots(1, 1, figsize=(6,5))
for cond in CONDITIONS.keys():
    ys = np.concatenate([p['y_true'] for p in regime_probs if p['condition']==cond])
    ps = np.concatenate([p['y_prob'] for p in regime_probs if p['condition']==cond])
    sp = selective_prediction_curve(ys, ps)
    ax.plot(sp['coverage'], sp['accuracy'], marker='o', label=cond)
ax.set_xlabel('Coverage')
ax.set_ylabel('Accuracy on selected set')
ax.set_title('Selective prediction (regime split)')
ax.legend()
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'selective_regime.png')
fig.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

# --- Rolling summaries barplot (ECE/NLL) ---
rolling_sum = pd.read_csv(os.path.join(TABLE_DIR, 'summary_ci95_step2.csv'))
roll = rolling_sum[rolling_sum['scenario']=='rolling'].copy()

fig, ax = plt.subplots(1, 1, figsize=(7,4))
x = np.arange(len(roll['condition']))
ax.bar(x, roll['ece_mean'].values)
ax.set_xticks(x)
ax.set_xticklabels(roll['condition'].values, rotation=45, ha='right')
ax.set_ylabel('ECE (mean across folds)')
ax.set_title('Calibration under rolling-window shift')
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, 'rolling_ece_bar.png')
fig.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)


In [ ]:
# Final: print output directory
print('All tables saved to:', TABLE_DIR)
print('All figures saved to:', FIG_DIR)


 Multi-ticker benchmark + Cross-market transfer


This stage extends evaluation beyond a single ticker to (i) a 20-ticker, multi-sector US benchmark and (ii) cross-market transfer (train on US tickers, test on UK/EU tickers).



In [ ]:
# =========================
# Step 3: Multi-ticker + Cross-market transfer
# =========================

# ---- Config ----
US_TICKERS = {
    'tech': ['AAPL', 'MSFT', 'NVDA', 'AMD'],
    'finance': ['JPM', 'BAC', 'GS', 'MS'],
    'energy': ['XOM', 'CVX', 'BP', 'SHEL'],
    'consumer': ['WMT', 'PG', 'KO', 'MCD'],
    'industrial': ['CAT', 'GE', 'HON', 'BA'],
}
ALL_US = sum(US_TICKERS.values(), [])

# Large, liquid UK/EU tickers (mix of London + major EU)
UK_EU_TICKERS = ['BP', 'HSBA', 'BARC', 'VOD', 'GSK', 'AZN', 'SAP', 'BMW.DE', 'DTE.DE', 'ENEL.MI', 'AIR.PA', 'SIEGY']

# Evaluation settings
SHOTS_OPT = SHOTS_BEST  # keep consistent with Step 1 optimum
TRAIN_START_US, TRAIN_END_US = TRAIN_START, TRAIN_END
TEST_END_US = TEST_END

# Multi-ticker rolling window (lighter repetitions for compute)
K_REPS_MULTI = max(3, K_REPS // 3)

print('US tickers:', len(ALL_US), ALL_US)
print('UK/EU tickers:', len(UK_EU_TICKERS), UK_EU_TICKERS)
print('K_REPS_MULTI:', K_REPS_MULTI, 'SHOTS_OPT:', SHOTS_OPT)

# ---- Helper: run a split for an arbitrary predictor (reuses Step 2 logic) ----

def run_single_split_for_predictor(predictor_obj, train_start, train_end, test_end, condition, rep_seed):
    X, y = predictor_obj.get_xy_by_dates(train_start, test_end)
    X_train, y_train, X_test, y_test = split_and_scale(X, y, train_end)
    y_train = _to_1d(y_train); y_test = _to_1d(y_test)

    if condition == 'noise_matched':
        # deterministic features
        res_det = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed, shots=None, mode='full')
        F_train_det = res_det.transform(X_train)
        F_test_det  = res_det.transform(X_test)

        # estimate per-feature variance induced by shots
        V_runs = []
        for s in range(5):
            res_s = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed + 1000 + s, shots=SHOTS_OPT, mode='full')
            F_train_s = res_s.transform(X_train)
            V_runs.append(np.var(F_train_s, axis=0, ddof=1))
        var_diag = np.mean(np.stack(V_runs, axis=0), axis=0)

        F_train, F_test = noise_matched_features(F_train_det, F_test_det, var_diag)
    else:
        cfg = CONDITIONS[condition]
        res = QuantumReservoir(n_qubits=N_QUBITS, seed=rep_seed, shots=cfg['shots'], mode=cfg['mode'])
        F_train = res.transform(X_train)
        F_test  = res.transform(X_test)

    metrics, y_prob = fit_and_eval(F_train, y_train, F_test, y_test, seed=rep_seed)
    y_prob = _to_1d(y_prob)
    return metrics, (y_test, y_prob)


In [ ]:
# ---- 3A: Multi-ticker rolling evaluation (US) ----

multi_rows = []

for ticker in tqdm(ALL_US, desc='Multi-ticker (US)'):
    pred = StockPredictor(ticker)

    # rolling folds based on same global date window
    folds = make_rolling_folds(TRAIN_START_US, TEST_END_US, TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS)

    for (tr_s, tr_e, te_e) in folds:
        for cond in CONDITIONS.keys():
            for k in range(K_REPS_MULTI):
                seed = 50000 + hash((ticker, tr_s, cond, k)) % 100000
                m, _ = run_single_split_for_predictor(pred, tr_s, tr_e, te_e, cond, seed)
                multi_rows.append({
                    'ticker': ticker,
                    'sector': next((sec for sec,ts in US_TICKERS.items() if ticker in ts), 'unknown'),
                    'scenario': 'multi_us_rolling',
                    'fold_train_start': tr_s,
                    'fold_train_end': tr_e,
                    'fold_test_end': te_e,
                    'condition': cond,
                    'rep': k,
                    **m
                })

multi_df = pd.DataFrame(multi_rows)
raw_path = os.path.join(TABLE_DIR, 'multi_ticker_us_rolling_raw.csv')
multi_df.to_csv(raw_path, index=False)
print('Saved:', raw_path)

# Summary: per-ticker mean over folds/reps + macro averages

per_ticker = (
    multi_df
    .groupby(['ticker','sector','condition'])[['accuracy','roc_auc','nll','brier','ece']]
    .mean()
    .reset_index()
)
per_ticker_path = os.path.join(TABLE_DIR, 'multi_ticker_us_per_ticker_mean.csv')
per_ticker.to_csv(per_ticker_path, index=False)
print('Saved:', per_ticker_path)

macro = (
    per_ticker
    .groupby(['condition'])[['accuracy','roc_auc','nll','brier','ece']]
    .agg(['mean','std'])
)
macro.columns = ['_'.join(c) for c in macro.columns]
macro = macro.reset_index()
macro_path = os.path.join(TABLE_DIR, 'multi_ticker_us_macro_mean_std.csv')
macro.to_csv(macro_path, index=False)
print('Saved:', macro_path)

# Win-rates (lower is better for ece/nll/brier; higher better for accuracy/auc)

def win_rate(metric, better='lower', a='quantum_full', b='noise_matched'):
    pv = per_ticker.pivot(index='ticker', columns='condition', values=metric)
    if better == 'lower':
        return float((pv[a] < pv[b]).mean())
    else:
        return float((pv[a] > pv[b]).mean())

win_table = pd.DataFrame({
    'metric': ['accuracy','roc_auc','nll','brier','ece'],
    'better': ['higher','higher','lower','lower','lower'],
})
win_table['winrate_q_vs_noise'] = [
    win_rate('accuracy','higher'),
    win_rate('roc_auc','higher'),
    win_rate('nll','lower'),
    win_rate('brier','lower'),
    win_rate('ece','lower'),
]
win_table['winrate_q_vs_no_ent'] = [
    float((per_ticker.pivot(index='ticker', columns='condition', values='accuracy')['quantum_full'] > per_ticker.pivot(index='ticker', columns='condition', values='accuracy')['no_entanglement']).mean()),
    float((per_ticker.pivot(index='ticker', columns='condition', values='roc_auc')['quantum_full'] > per_ticker.pivot(index='ticker', columns='condition', values='roc_auc')['no_entanglement']).mean()),
    float((per_ticker.pivot(index='ticker', columns='condition', values='nll')['quantum_full'] < per_ticker.pivot(index='ticker', columns='condition', values='nll')['no_entanglement']).mean()),
    float((per_ticker.pivot(index='ticker', columns='condition', values='brier')['quantum_full'] < per_ticker.pivot(index='ticker', columns='condition', values='brier')['no_entanglement']).mean()),
    float((per_ticker.pivot(index='ticker', columns='condition', values='ece')['quantum_full'] < per_ticker.pivot(index='ticker', columns='condition', values='ece')['no_entanglement']).mean()),
]
win_table_path = os.path.join(TABLE_DIR, 'multi_ticker_winrates.csv')
win_table.to_csv(win_table_path, index=False)
print('Saved:', win_table_path)

# Paired significance test across tickers (Wilcoxon signed-rank)
from scipy.stats import wilcoxon

def paired_wilcoxon(metric, better='lower', a='quantum_full', b='noise_matched'):
    pv = per_ticker.pivot(index='ticker', columns='condition', values=metric).dropna()
    da = pv[a].values; db = pv[b].values
    # Wilcoxon tests median of differences; use alternative consistent with better direction
    if better == 'lower':
        stat, p = wilcoxon(db - da, alternative='greater')  # tests a < b
    else:
        stat, p = wilcoxon(da - db, alternative='greater')  # tests a > b
    return float(p)

ptest = {
    'metric': ['accuracy','roc_auc','nll','brier','ece'],
    'p_q_vs_noise': [
        paired_wilcoxon('accuracy','higher'),
        paired_wilcoxon('roc_auc','higher'),
        paired_wilcoxon('nll','lower'),
        paired_wilcoxon('brier','lower'),
        paired_wilcoxon('ece','lower'),
    ]
}
ptest_df = pd.DataFrame(ptest)
ptest_path = os.path.join(TABLE_DIR, 'multi_ticker_pvalues_wilcoxon.csv')
ptest_df.to_csv(ptest_path, index=False)
print('Saved:', ptest_path)

# ---- Visualisations: boxplots across tickers ----

import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
# Boxplot without seaborn to keep dependencies light
conds = list(CONDITIONS.keys())
vals = [per_ticker[per_ticker['condition']==c]['ece'].values for c in conds]
plt.boxplot(vals, labels=conds)
plt.ylabel('ECE (per-ticker mean; lower is better)')
plt.title('Calibration across 20 US tickers (rolling evaluation)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'multi_ticker_us_ece_box.png')
plt.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

plt.figure(figsize=(8,4))
vals = [per_ticker[per_ticker['condition']==c]['accuracy'].values for c in conds]
plt.boxplot(vals, labels=conds)
plt.ylabel('Accuracy (per-ticker mean)')
plt.title('Accuracy across 20 US tickers (rolling evaluation)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'multi_ticker_us_acc_box.png')
plt.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

# ---- 3B: Cross-market transfer (train US, test UK/EU) ----

# Train readout on aggregated US training data (single regime split for transfer).
# We fit a single StandardScaler on aggregated US raw indicators (train period) and reuse it for all markets
# to avoid peeking into the target-market distribution.

TRANSFER_TRAIN_START, TRANSFER_TRAIN_END = TRAIN_START_US, TRAIN_END_US
TRANSFER_TEST_START, TRANSFER_TEST_END = TRAIN_END_US, TEST_END_US

# ---- Fit global scaler on aggregated US raw features ----
X_raw_list = []
y_raw_list = []
for ticker in ALL_US:
    pred = StockPredictor(ticker)
    X_raw, y_raw = pred.get_xy_by_dates(TRANSFER_TRAIN_START, TRANSFER_TRAIN_END)
        if X_raw is None or len(X_raw)==0 or y_raw is None or len(y_raw)==0:
            print(f'[WARN] No training samples for {ticker} in transfer train window; skipping.')
            continue
    X_raw_list.append(X_raw.values)
    y_raw_list.append(_to_1d(y_raw.values))
if len(X_raw_list)==0:
    raise ValueError('No US training data collected for global scaler. Check date ranges and tickers.')
X_raw_all = np.vstack(X_raw_list)
scaler_global = StandardScaler()
scaler_global.fit(X_raw_all)

# ---- Feature builders using the global scaler ----
def build_reservoir_features_from_raw(X_raw, condition, seed):
    # Guard against empty inputs (some tickers may have no data in a given window)
    if X_raw is None or len(X_raw)==0:
        return None
    Xs = scaler_global.transform(X_raw)
    if condition == 'noise_matched':
        res_det = QuantumReservoir(n_qubits=N_QUBITS, seed=seed, shots=None, mode='full')
        F_det = res_det.transform(Xs)
        # Estimate diagonal variance of finite-shot features on the same inputs
        V_runs = []
        for s in range(5):
            res_s = QuantumReservoir(n_qubits=N_QUBITS, seed=seed+1000+s, shots=SHOTS_OPT, mode='full')
            F_s = res_s.transform(Xs)
            V_runs.append(np.var(F_s, axis=0, ddof=1))
        var_diag = np.mean(np.stack(V_runs, axis=0), axis=0)
        eps = np.random.normal(scale=np.sqrt(np.maximum(var_diag,1e-12)), size=F_det.shape)
        return F_det + eps
    else:
        cfg = CONDITIONS[condition]
        res = QuantumReservoir(n_qubits=N_QUBITS, seed=seed, shots=cfg['shots'], mode=cfg['mode'])
        return res.transform(Xs)

transfer_rows = []

for cond in CONDITIONS.keys():
    # Aggregate US training reservoir features
    F_list = []
    y_list = []
    for ti, ticker in enumerate(ALL_US):
        pred = StockPredictor(ticker)
        X_raw, y_raw = pred.get_xy_by_dates(TRANSFER_TRAIN_START, TRANSFER_TRAIN_END)
        if X_raw is None or len(X_raw)==0 or y_raw is None or len(y_raw)==0:
            print(f'[WARN] No training samples for {ticker} in transfer train window; skipping.')
            continue
        F = build_reservoir_features_from_raw(X_raw.values, cond, seed=70000 + 31*ti)
        if F is None or len(F)==0:
            print(f'[WARN] Empty features for {ticker} during transfer training; skipping.')
            continue
        F_list.append(F)
        y_list.append(_to_1d(y_raw.values))
    F_train = np.vstack(F_list)
    y_train = np.concatenate(y_list)

    clf = LogisticRegression(max_iter=2000, random_state=42)
    clf.fit(F_train, y_train)

    # Evaluate on each UK/EU ticker (no retraining)
    for tj, ticker in enumerate(UK_EU_TICKERS):
        pred_te = StockPredictor(ticker)
        X_raw_te, y_raw_te = pred_te.get_xy_by_dates(TRANSFER_TEST_START, TRANSFER_TEST_END)
        if X_raw_te is None or len(X_raw_te)==0 or y_raw_te is None or len(y_raw_te)==0:
            print(f'[WARN] No test samples for {ticker} in transfer test window; skipping.')
            continue
        F_te = build_reservoir_features_from_raw(X_raw_te.values, cond, seed=80000 + 17*tj)
        if F_te is None or len(F_te)==0:
            print(f'[WARN] Empty features for {ticker} during transfer test; skipping.')
            continue
        y_test = _to_1d(y_raw_te.values)
        y_prob = clf.predict_proba(F_te)[:, 1]
        m = compute_metrics(y_test, y_prob)
        transfer_rows.append({'condition': cond, 'ticker': ticker, **m})

transfer_df = pd.DataFrame(transfer_rows)
transfer_path = os.path.join(TABLE_DIR, 'cross_market_transfer_us_to_ukeu.csv')
transfer_df.to_csv(transfer_path, index=False)
print('Saved:', transfer_path)

# Plot mean ECE across UK/EU tickers (lower is better)
mean_ece = transfer_df.groupby('condition')['ece'].mean().reindex(list(CONDITIONS.keys()))
plt.figure(figsize=(7,4))
plt.bar(mean_ece.index, mean_ece.values)
plt.ylabel('ECE (mean across UK/EU tickers)')
plt.title('Cross-market transfer: train US (2022) → test UK/EU (2023–2024)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'cross_market_transfer_mean_ece.png')
plt.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

# Also save a per-ticker ECE boxplot for transfer
plt.figure(figsize=(8,4))
conds = list(CONDITIONS.keys())
vals = [transfer_df[transfer_df['condition']==c]['ece'].values for c in conds]
plt.boxplot(vals, labels=conds)
plt.ylabel('ECE (per-ticker; lower is better)')
plt.title('Cross-market transfer calibration across UK/EU tickers')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, 'cross_market_transfer_ece_box.png')
plt.savefig(fig_path, dpi=200)
plt.show()
print('Saved:', fig_path)

print('Step 3 complete. Tables in', TABLE_DIR, 'Figures in', FIG_DIR)

In [ ]:
TRANSFER_TRAIN_START, TRANSFER_TRAIN_END = TRAIN_START_US, TRAIN_END_US
TRANSFER_TEST_START, TRANSFER_TEST_END = TRAIN_END_US, TEST_END_US

Strong classical & uncertainty-aware baselines
baselines with matched uncertainty budgets, feature dimensions, and post-hoc temperature scaling for calibration fairness.

In [ ]:

# =========================
# Step 4: Baseline suite
# =========================
# Baselines included:
#  - Logistic regression on raw indicators
#  - Random Fourier Features (RFF) + logistic
#  - Random MLP features (frozen) + logistic
#  - ESN reservoir + logistic
#  - Ensemble ESN (size-matched) + prob-averaging
#  - Deep ensemble (tiny MLP) on raw indicators
#  - MC-dropout tiny MLP on raw indicators
#  - Temperature scaling (post-hoc) applied to ALL methods for fair calibration comparison
#


import numpy as np
import pandas as pd
import os
from dataclasses import dataclass
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.kernel_approximation import RBFSampler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Torch for MC-dropout baseline (lightweight)
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("[WARN] PyTorch not available; MC-dropout baseline will be skipped.", e)

# -------------------------
# Global metrics helper (ensure defined)
# -------------------------
if 'compute_metrics' not in globals():
    from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss

    def compute_metrics(y_true, y_prob, n_bins=10):
        y_true = np.asarray(y_true).reshape(-1)
        y_prob = np.asarray(y_prob).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)
        acc = accuracy_score(y_true, y_pred)
        auc = roc_auc_score(y_true, y_prob)
        nll = log_loss(y_true, y_prob, labels=[0, 1])
        brier = brier_score_loss(y_true, y_prob)

        bins = np.linspace(0.0, 1.0, n_bins + 1)
        bin_ids = np.digitize(y_prob, bins) - 1
        ece = 0.0
        for b in range(n_bins):
            mask = bin_ids == b
            if np.any(mask):
                conf = y_prob[mask].mean()
                acc_bin = y_true[mask].mean()
                ece += np.abs(acc_bin - conf) * (mask.sum() / len(y_true))
        return {"accuracy": float(acc), "roc_auc": float(auc), "nll": float(nll), "brier": float(brier), "ece": float(ece)}

# -------------------------
# Temperature scaling (post-hoc calibration)
# -------------------------
def _logit(p, eps=1e-6):
    p = np.clip(np.asarray(p), eps, 1 - eps)
    return np.log(p / (1 - p))

def _sigmoid(x):
    return 1 / (1 + np.exp(-x))

def fit_temperature_from_logits(logits_val, y_val, T_grid=None):
    """
    Fit scalar temperature T by minimizing NLL on validation set.
    """
    y_val = np.asarray(y_val).reshape(-1)
    logits_val = np.asarray(logits_val).reshape(-1)
    if T_grid is None:
        # Log-spaced grid is robust and fast
        T_grid = np.exp(np.linspace(np.log(0.5), np.log(10.0), 60))
    best_T, best_nll = 1.0, float("inf")
    for T in T_grid:
        p = _sigmoid(logits_val / T)
        nll = log_loss(y_val, np.vstack([1 - p, p]).T, labels=[0, 1])
        if nll < best_nll:
            best_nll, best_T = nll, T
    return float(best_T), float(best_nll)

def apply_temperature_to_proba(p, T):
    logits = _logit(p)
    return _sigmoid(logits / T)

def chrono_train_val_split(X, y, val_frac=0.2):
    n = len(X)
    if n < 10:
        return X, y, X, y  # fallback
    cut = int(np.floor(n * (1 - val_frac)))
    return X[:cut], y[:cut], X[cut:], y[cut:]

# -------------------------
# Random MLP (frozen) feature map
# -------------------------
def random_mlp_features(X, out_dim, seed=0, hidden_dim=128):
    rng = np.random.RandomState(seed)
    W1 = rng.normal(scale=1.0 / np.sqrt(X.shape[1]), size=(X.shape[1], hidden_dim))
    b1 = rng.normal(scale=0.1, size=(hidden_dim,))
    H = np.maximum(0.0, X @ W1 + b1)  # ReLU
    W2 = rng.normal(scale=1.0 / np.sqrt(hidden_dim), size=(hidden_dim, out_dim))
    b2 = rng.normal(scale=0.1, size=(out_dim,))
    return H @ W2 + b2

# -------------------------
# ESN (Echo State Network) baseline
# -------------------------
@dataclass
class ESNConfig:
    n_reservoir: int = 128
    spectral_radius: float = 0.9
    sparsity: float = 0.9
    leak: float = 1.0
    input_scale: float = 1.0
    ridge_alpha: float = 1e-3  # not used for logistic, kept for extensions

class ESN:
    def __init__(self, input_dim, cfg: ESNConfig, seed=0):
        self.cfg = cfg
        rng = np.random.RandomState(seed)
        N = cfg.n_reservoir

        # Input weights
        self.Win = (rng.uniform(-1, 1, size=(N, input_dim)) * cfg.input_scale).astype(np.float32)

        # Reservoir weights with sparsity
        W = rng.uniform(-1, 1, size=(N, N)).astype(np.float32)
        mask = rng.rand(N, N) < (1.0 - cfg.sparsity)
        W *= mask.astype(np.float32)

        # Scale to spectral radius
        # Use power iteration (fast) instead of full eig
        v = rng.normal(size=(N,)).astype(np.float32)
        for _ in range(50):
            v = W @ v
            norm = np.linalg.norm(v) + 1e-12
            v = v / norm
        approx_sr = np.linalg.norm(W @ v) / (np.linalg.norm(v) + 1e-12)
        if approx_sr > 0:
            W *= (cfg.spectral_radius / approx_sr)
        self.W = W
        self.state = np.zeros((N,), dtype=np.float32)

    def reset(self):
        self.state[:] = 0.0

    def transform(self, X):
        # X: (T, input_dim), sequential update
        X = np.asarray(X, dtype=np.float32)
        states = np.zeros((len(X), self.cfg.n_reservoir), dtype=np.float32)
        for t in range(len(X)):
            u = X[t]
            pre = self.W @ self.state + self.Win @ u
            x_new = np.tanh(pre)
            if self.cfg.leak < 1.0:
                self.state = (1 - self.cfg.leak) * self.state + self.cfg.leak * x_new
            else:
                self.state = x_new
            states[t] = self.state
        return states

# -------------------------
# Tiny PyTorch MLP for MC-dropout (optional)
# -------------------------
class TinyDropoutMLP(nn.Module):
    def __init__(self, d_in, hidden=32, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_torch_mlp(X_train, y_train, X_val, y_val, hidden=32, p=0.2, lr=1e-3, epochs=80, wd=1e-4, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    d_in = X_train.shape[1]
    model = TinyDropoutMLP(d_in, hidden=hidden, p=p)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.BCEWithLogitsLoss()

    Xtr = torch.tensor(X_train, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.float32)
    Xva = torch.tensor(X_val, dtype=torch.float32)
    yva = torch.tensor(y_val, dtype=torch.float32)

    best_state, best_val = None, float("inf")
    for ep in range(epochs):
        model.train()
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, ytr)
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_loss = loss_fn(val_logits, yva).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def mc_dropout_predict_proba(model, X, mc_passes=50):
    Xte = torch.tensor(X, dtype=torch.float32)
    probs = []
    model.train()  # keep dropout ON
    with torch.no_grad():
        for _ in range(mc_passes):
            logits = model(Xte).cpu().numpy()
            probs.append(1 / (1 + np.exp(-logits)))
    p = np.mean(np.stack(probs, axis=0), axis=0)
    return p

# -------------------------
# Baseline runner for a single ticker and fold
# -------------------------
def run_fold_baselines(predictor, train_start, train_end, test_end, baseline_name, seed=0,
                       ensemble_M=5, feature_dim=64, mc_passes=50):
    """
    Returns metrics dict with calibrated probabilities (temperature scaling) and uncalibrated.
    """
    X_raw_tr, y_raw_tr = predictor.get_xy_by_dates(train_start, train_end)
    X_raw_te, y_raw_te = predictor.get_xy_by_dates(train_end, test_end)

    if X_raw_tr is None or len(X_raw_tr) < 30 or X_raw_te is None or len(X_raw_te) < 10:
        return None

    X_tr = X_raw_tr.values
    y_tr = _to_1d(y_raw_tr.values)
    X_te = X_raw_te.values
    y_te = _to_1d(y_raw_te.values)

    # chrono val split inside train
    X_tr_sub, y_tr_sub, X_val, y_val = chrono_train_val_split(X_tr, y_tr, val_frac=0.2)

    # scaler fit only on training subset (no leakage)
    scaler = StandardScaler()
    scaler.fit(X_tr_sub)
    Xtr_s = scaler.transform(X_tr_sub)
    Xva_s = scaler.transform(X_val)
    Xte_s = scaler.transform(X_te)

    # Helper to fit temp scaling using validation logits
    def calibrate_from_proba(p_val, p_test):
        logits_val = _logit(p_val)
        T, _ = fit_temperature_from_logits(logits_val, y_val)
        p_test_cal = apply_temperature_to_proba(p_test, T)
        return p_test_cal, T

    # ------------- Baselines -------------
    if baseline_name == "lr_raw":
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(Xtr_s, y_tr_sub)
        p_val = clf.predict_proba(Xva_s)[:, 1]
        p_te = clf.predict_proba(Xte_s)[:, 1]
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "rff_lr":
        # RFF mapping + logistic readout; match feature_dim
        rff = RBFSampler(gamma=1.0, n_components=feature_dim, random_state=seed)
        Ztr = rff.fit_transform(Xtr_s)
        Zva = rff.transform(Xva_s)
        Zte = rff.transform(Xte_s)
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(Ztr, y_tr_sub)
        p_val = clf.predict_proba(Zva)[:, 1]
        p_te = clf.predict_proba(Zte)[:, 1]
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "rand_mlp_feat":
        # Frozen random MLP features -> logistic
        Ztr = random_mlp_features(Xtr_s, out_dim=feature_dim, seed=seed, hidden_dim=max(64, feature_dim))
        Zva = random_mlp_features(Xva_s, out_dim=feature_dim, seed=seed, hidden_dim=max(64, feature_dim))
        Zte = random_mlp_features(Xte_s, out_dim=feature_dim, seed=seed, hidden_dim=max(64, feature_dim))
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(Ztr, y_tr_sub)
        p_val = clf.predict_proba(Zva)[:, 1]
        p_te = clf.predict_proba(Zte)[:, 1]
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "esn_single":
        # ESN state dimension matched to feature_dim
        cfg = ESNConfig(n_reservoir=feature_dim, spectral_radius=0.9, sparsity=0.9, leak=1.0, input_scale=1.0)
        esn = ESN(input_dim=Xtr_s.shape[1], cfg=cfg, seed=seed)
        esn.reset()
        Ftr = esn.transform(Xtr_s)
        esn.reset()
        Fva = esn.transform(Xva_s)
        esn.reset()
        Fte = esn.transform(Xte_s)

        clf = LogisticRegression(max_iter=2000, random_state=seed)
        clf.fit(Ftr, y_tr_sub)
        p_val = clf.predict_proba(Fva)[:, 1]
        p_te = clf.predict_proba(Fte)[:, 1]
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "esn_ens":
        # Ensemble of ESNs, size matched by ensemble_M
        cfg = ESNConfig(n_reservoir=feature_dim, spectral_radius=0.9, sparsity=0.9, leak=1.0, input_scale=1.0)
        p_val_list = []
        p_te_list = []
        for m in range(ensemble_M):
            esn = ESN(input_dim=Xtr_s.shape[1], cfg=cfg, seed=seed + 100 * m)
            esn.reset(); Ftr = esn.transform(Xtr_s)
            esn.reset(); Fva = esn.transform(Xva_s)
            esn.reset(); Fte = esn.transform(Xte_s)
            clf = LogisticRegression(max_iter=2000, random_state=seed + 100 * m)
            clf.fit(Ftr, y_tr_sub)
            p_val_list.append(clf.predict_proba(Fva)[:, 1])
            p_te_list.append(clf.predict_proba(Fte)[:, 1])
        p_val = np.mean(np.stack(p_val_list, axis=0), axis=0)
        p_te = np.mean(np.stack(p_te_list, axis=0), axis=0)
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "deep_ens_mlp":
        # Deep ensemble of small MLPs (sklearn), prob-averaging
        p_val_list = []
        p_te_list = []
        for m in range(ensemble_M):
            mlp = MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-4, max_iter=400,
                                early_stopping=True, n_iter_no_change=15,
                                random_state=seed + 77 * m)
            mlp.fit(Xtr_s, y_tr_sub)
            p_val_list.append(mlp.predict_proba(Xva_s)[:, 1])
            p_te_list.append(mlp.predict_proba(Xte_s)[:, 1])
        p_val = np.mean(np.stack(p_val_list, axis=0), axis=0)
        p_te = np.mean(np.stack(p_te_list, axis=0), axis=0)
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    elif baseline_name == "mc_dropout_mlp":
        if not TORCH_OK:
            return None
        # Single trained MLP with dropout; MC sampling at test
        model = train_torch_mlp(Xtr_s, y_tr_sub, Xva_s, y_val, hidden=32, p=0.2, lr=1e-3, epochs=120, seed=seed)
        # For calibration: approximate val probs using MC passes too
        p_val = mc_dropout_predict_proba(model, Xva_s, mc_passes=mc_passes)
        p_te = mc_dropout_predict_proba(model, Xte_s, mc_passes=mc_passes)
        p_te_cal, T = calibrate_from_proba(p_val, p_te)

    else:
        raise ValueError(f"Unknown baseline: {baseline_name}")

    # metrics: uncalibrated + calibrated
    m_uncal = compute_metrics(y_te, p_te)
    m_cal = compute_metrics(y_te, p_te_cal)

    out = {"T": float(T)}
    for k, v in m_uncal.items():
        out[f"{k}_uncal"] = v
    for k, v in m_cal.items():
        out[f"{k}_cal"] = v
    return out

# -------------------------
# Run Step-4 evaluation on multi-ticker rolling protocol (US)
# -------------------------
BASELINES = [
    "lr_raw",
    "rff_lr",
    "rand_mlp_feat",
    "esn_single",
    "esn_ens",
    "deep_ens_mlp",
    "mc_dropout_mlp",
]

# Match "uncertainty budget"
ENSEMBLE_M = 5         # match classical ensembles / deep ensembles
MC_PASSES = int(SHOTS_OPT) if 'SHOTS_OPT' in globals() and np.isfinite(SHOTS_OPT) else 200  # comparable to shots
FEATURE_DIM = 64       # keep lightweight; we also log this (can be set to match quantum feature dim if known)

# Use same overall date range as multi-ticker rolling evaluation
START_ALL = TRAIN_START_US
END_ALL = TEST_END_US

folds = make_rolling_folds(START_ALL, END_ALL, train_months=12, test_months=3, step_months=3)
print("Rolling folds:", len(folds))

rows = []
for ticker in tqdm(ALL_US, desc="Step4 multi-ticker"):
    pred = StockPredictor(ticker)
    for (tr_s, tr_e, te_e) in folds:
        for b in BASELINES:
            # Repeat a few seeds for stochastic baselines (kept small for runtime)
            for rep in range(3):
                seed = 9000 + 97 * rep
                m = run_fold_baselines(pred, tr_s, tr_e, te_e, b, seed=seed,
                                       ensemble_M=ENSEMBLE_M, feature_dim=FEATURE_DIM, mc_passes=MC_PASSES)
                if m is None:
                    continue
                rows.append({
                    "ticker": ticker, "fold_train_start": tr_s, "fold_train_end": tr_e, "fold_test_end": te_e,
                    "baseline": b, "rep": rep, **m
                })

step4_df = pd.DataFrame(rows)
step4_raw_path = os.path.join(TABLE_DIR, "step4_baselines_multi_ticker_raw.csv")
step4_df.to_csv(step4_raw_path, index=False)
print("Saved:", step4_raw_path)

# Aggregate per ticker (mean over folds+reps)
agg_cols = [c for c in step4_df.columns if c.endswith("_cal")]
per_ticker = step4_df.groupby(["ticker", "baseline"])[agg_cols].mean().reset_index()
per_ticker_path = os.path.join(TABLE_DIR, "step4_baselines_multi_ticker_per_ticker.csv")
per_ticker.to_csv(per_ticker_path, index=False)
print("Saved:", per_ticker_path)

# Macro summary across tickers
macro = per_ticker.groupby("baseline")[agg_cols].agg(["mean", "std"]).reset_index()
macro_path = os.path.join(TABLE_DIR, "step4_baselines_multi_ticker_macro.csv")
macro.to_csv(macro_path, index=False)
print("Saved:", macro_path)

# -------------------------
# Visualisations (boxplots across tickers) for calibrated metrics
# -------------------------
def _boxplot_metric(metric_key, title, fname):
    plt.figure(figsize=(11, 4))
    order = BASELINES
    vals = [per_ticker[per_ticker["baseline"] == b][metric_key].values for b in order]
    plt.boxplot(vals, labels=order, showfliers=True)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric_key.replace("_cal", "").upper())
    plt.title(title)
    plt.tight_layout()
    out = os.path.join(FIG_DIR, fname)
    plt.savefig(out, dpi=200)
    plt.show()
    print("Saved:", out)

_boxplot_metric("accuracy_cal", "Accuracy across 20 US tickers (Step 4 baselines; temperature-scaled)", "step4_us_accuracy_box.png")
_boxplot_metric("ece_cal", "Calibration (ECE) across 20 US tickers (Step 4 baselines; temperature-scaled)", "step4_us_ece_box.png")
_boxplot_metric("nll_cal", "NLL across 20 US tickers (Step 4 baselines; temperature-scaled)", "step4_us_nll_box.png")



In [ ]:
# -------------------------
# Cross-market transfer baselines (US->UK/EU) with the SAME global scaler fitted on US train
# -------------------------
print("\n[Step4] Cross-market transfer baselines (US -> UK/EU)")

# Fit global scaler on US train ONLY
X_us_list = []
y_us_list = []
for ticker in ALL_US:
    pred = StockPredictor(ticker)
    X_raw, y_raw = pred.get_xy_by_dates(TRANSFER_TRAIN_START, TRANSFER_TRAIN_END)
    if X_raw is None or len(X_raw) == 0:
        continue
    X_us_list.append(X_raw.values)
    y_us_list.append(_to_1d(y_raw.values))
X_us_all = np.vstack(X_us_list)
scaler_global = StandardScaler().fit(X_us_all)

def build_X_scaled(predictor, start, end):
    X_raw, y_raw = predictor.get_xy_by_dates(start, end)
    if X_raw is None or len(X_raw) == 0 or y_raw is None or len(y_raw) == 0:
        return None, None
    Xs = scaler_global.transform(X_raw.values)
    y = _to_1d(y_raw.values)
    return Xs, y

transfer_rows = []
for b in BASELINES:
    # Train on aggregated US
    Xtr_list, ytr_list = [], []
    for ticker in ALL_US:
        Xs, y = build_X_scaled(StockPredictor(ticker), TRANSFER_TRAIN_START, TRANSFER_TRAIN_END)
        if Xs is None:
            continue
        Xtr_list.append(Xs); ytr_list.append(y)
    if len(Xtr_list) == 0:
        continue
    Xtr = np.vstack(Xtr_list); ytr = np.concatenate(ytr_list)

    # chrono val split for temp scaling
    Xtr_sub, ytr_sub, Xva, yva = chrono_train_val_split(Xtr, ytr, val_frac=0.2)

    # Train baseline on US
    if b == "lr_raw":
        model = LogisticRegression(max_iter=2000, random_state=123)
        model.fit(Xtr_sub, ytr_sub)
        p_val = model.predict_proba(Xva)[:, 1]
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            p = model.predict_proba(Xte)[:, 1]
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "rff_lr":
        rff = RBFSampler(gamma=1.0, n_components=FEATURE_DIM, random_state=123)
        Ztr = rff.fit_transform(Xtr_sub)
        Zva = rff.transform(Xva)
        model = LogisticRegression(max_iter=2000, random_state=123)
        model.fit(Ztr, ytr_sub)
        p_val = model.predict_proba(Zva)[:, 1]
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            Zte = rff.transform(Xte)
            p = model.predict_proba(Zte)[:, 1]
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "rand_mlp_feat":
        Ztr = random_mlp_features(Xtr_sub, out_dim=FEATURE_DIM, seed=123, hidden_dim=max(64, FEATURE_DIM))
        Zva = random_mlp_features(Xva, out_dim=FEATURE_DIM, seed=123, hidden_dim=max(64, FEATURE_DIM))
        model = LogisticRegression(max_iter=2000, random_state=123)
        model.fit(Ztr, ytr_sub)
        p_val = model.predict_proba(Zva)[:, 1]
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            Zte = random_mlp_features(Xte, out_dim=FEATURE_DIM, seed=123, hidden_dim=max(64, FEATURE_DIM))
            p = model.predict_proba(Zte)[:, 1]
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "esn_single":
        cfg = ESNConfig(n_reservoir=FEATURE_DIM)
        esn = ESN(input_dim=Xtr_sub.shape[1], cfg=cfg, seed=123)
        esn.reset(); Ftr = esn.transform(Xtr_sub)
        esn.reset(); Fva = esn.transform(Xva)
        model = LogisticRegression(max_iter=2000, random_state=123)
        model.fit(Ftr, ytr_sub)
        p_val = model.predict_proba(Fva)[:, 1]
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            esn.reset(); Fte = esn.transform(Xte)
            p = model.predict_proba(Fte)[:, 1]
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "esn_ens":
        cfg = ESNConfig(n_reservoir=FEATURE_DIM)
        # Train ensemble models on US
        esn_models = []
        for m in range(ENSEMBLE_M):
            esn = ESN(input_dim=Xtr_sub.shape[1], cfg=cfg, seed=123 + 10*m)
            esn.reset(); Ftr = esn.transform(Xtr_sub)
            model = LogisticRegression(max_iter=2000, random_state=123 + 10*m)
            model.fit(Ftr, ytr_sub)
            esn_models.append((esn, model))

        # Temp scaling using ensemble val probs
        p_val_list = []
        for (esn, model) in esn_models:
            esn.reset(); Fva = esn.transform(Xva)
            p_val_list.append(model.predict_proba(Fva)[:, 1])
        p_val = np.mean(np.stack(p_val_list, axis=0), axis=0)
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            p_list = []
            for (esn, model) in esn_models:
                esn.reset(); Fte = esn.transform(Xte)
                p_list.append(model.predict_proba(Fte)[:, 1])
            p = np.mean(np.stack(p_list, axis=0), axis=0)
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "deep_ens_mlp":
        # Deep ensemble (sklearn) on US
        models = []
        for m in range(ENSEMBLE_M):
            mlp = MLPClassifier(hidden_layer_sizes=(32,), alpha=1e-4, max_iter=400,
                                early_stopping=True, n_iter_no_change=15,
                                random_state=123 + 33*m)
            mlp.fit(Xtr_sub, ytr_sub)
            models.append(mlp)
        p_val_list = [m.predict_proba(Xva)[:, 1] for m in models]
        p_val = np.mean(np.stack(p_val_list, axis=0), axis=0)
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            p_list = [m.predict_proba(Xte)[:, 1] for m in models]
            p = np.mean(np.stack(p_list, axis=0), axis=0)
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

    elif b == "mc_dropout_mlp":
        if not TORCH_OK:
            continue
        model = train_torch_mlp(Xtr_sub, ytr_sub, Xva, yva, hidden=32, p=0.2, lr=1e-3, epochs=120, seed=123)
        p_val = mc_dropout_predict_proba(model, Xva, mc_passes=MC_PASSES)
        T, _ = fit_temperature_from_logits(_logit(p_val), yva)

        for ticker in UK_EU_TICKERS:
            Xte, yte = build_X_scaled(StockPredictor(ticker), TRANSFER_TEST_START, TRANSFER_TEST_END)
            if Xte is None:
                continue
            p = mc_dropout_predict_proba(model, Xte, mc_passes=MC_PASSES)
            p_cal = apply_temperature_to_proba(p, T)
            m = compute_metrics(yte, p_cal)
            transfer_rows.append({"baseline": b, "ticker": ticker, **m})

transfer_df = pd.DataFrame(transfer_rows)
transfer_path = os.path.join(TABLE_DIR, "step4_cross_market_transfer_baselines.csv")
transfer_df.to_csv(transfer_path, index=False)
print("Saved:", transfer_path)

# Visualisations: mean ECE across UK/EU tickers
mean_ece = transfer_df.groupby("baseline")["ece"].mean().reindex(BASELINES)
plt.figure(figsize=(10, 4))
plt.bar(mean_ece.index, mean_ece.values)
plt.ylabel("ECE (mean across UK/EU tickers; lower is better)")
plt.title("Step 4 baselines: cross-market transfer calibration (train US → test UK/EU)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
out = os.path.join(FIG_DIR, "step4_cross_market_mean_ece.png")
plt.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

# Boxplot of per-ticker ECE
plt.figure(figsize=(11, 4))
vals = [transfer_df[transfer_df["baseline"] == b]["ece"].values for b in BASELINES]
plt.boxplot(vals, labels=BASELINES, showfliers=True)
plt.ylabel("ECE (per-ticker; lower is better)")
plt.title("Step 4 baselines: cross-market transfer (US→UK/EU) calibration")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
out = os.path.join(FIG_DIR, "step4_cross_market_ece_box.png")
plt.savefig(out, dpi=200)
plt.show()
print("Saved:", out)

print("Step 4 complete. Tables in", TABLE_DIR, "Figures in", FIG_DIR)
